In [1]:
import torchxrayvision as xrv
import torch, torchvision
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from matplotlib.lines import Line2D
import umap.umap_ as umap

from sklearn.manifold import Isomap
import pacmap

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_model():
    model = xrv.models.DenseNet(weights="densenet121-res224-chex")
    model.eval()
    return model.to(DEVICE)

def load_dataset():
    transform = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(),xrv.datasets.XRayResizer(224)])
    d_nih = xrv.datasets.NIH_Google_Dataset(imgpath="./nih_data_small", transform=transform)
    print(f'found {len(d_nih)} images')
    return d_nih


In [2]:
# Fixed seed for reproducibility
RANDOM_STATE = 42

In [3]:
# Pathologies to investigate
#TODO: think about adding the other Pathologies as well
#TODO: remove Nodule/Mass
pathologies = [
    "Lung Opacity",
    "Fracture",
    #"Nodule/Mass",
    "Pneumothorax"
]

In [4]:
# ============================================================
# Load model + dataset and extract:
#   1. model embeddings
#   2. original dataset image representations
#   3. model predictions
#   4. labels
# ============================================================

model = load_model()
dataset = load_dataset()
model.eval()

dataset_pathologies = list(dataset.pathologies)

features_list = []
image_features_list = []
labels_list = []
predictions_list = []


# Capture features entering DenseNet's final classifier.
def capture_features(module, inputs):
    features_list.append(
        inputs[0].detach().cpu().numpy()
    )


hook = model.classifier.register_forward_pre_hook(
    capture_features
)


with torch.no_grad():

    for i in range(len(dataset)):

        sample = dataset[i]

        # Preprocessed X-ray image
        img = np.asarray(sample["img"]).squeeze()

        # Store the actual dataset image as the high-dimensional
        # representation for the "Dataset image" feature space.
        #
        # Example:
        # 224 x 224 image -> 50,176-dimensional vector
        image_features_list.append(
            img.astype(np.float32).reshape(-1)
        )

        image = torch.as_tensor(
            sample["img"],
            dtype=torch.float32
        ).unsqueeze(0).to(DEVICE)

        # Model prediction
        output = model(image)

        predictions_list.append(
            output.detach().cpu().numpy().reshape(-1)
        )

        labels_list.append(
            np.asarray(sample["lab"]).reshape(-1)
        )


hook.remove()


# ------------------------------------------------------------
# Convert everything to matrices
# ------------------------------------------------------------

features = np.concatenate(
    features_list,
    axis=0
)

image_features = np.stack(
    image_features_list,
    axis=0
)

labels = np.stack(
    labels_list,
    axis=0
)

predictions = np.stack(
    predictions_list,
    axis=0
)


print("Model feature matrix:  ", features.shape)
print("Dataset image matrix:  ", image_features.shape)
print("NIH labels:             ", labels.shape)
print("DenseNet outputs:       ", predictions.shape)

print("\nModel pathology labels:")
print(model.pathologies)

found 810 images
Model feature matrix:   (810, 1024)
Dataset image matrix:   (810, 50176)
NIH labels:              (810, 4)
DenseNet outputs:        (810, 18)

Model pathology labels:
['Atelectasis', 'Consolidation', '', 'Pneumothorax', 'Edema', '', '', 'Effusion', 'Pneumonia', '', 'Cardiomegaly', '', '', '', 'Lung Lesion', 'Fracture', 'Lung Opacity', 'Enlarged Cardiomediastinum']


In [5]:
dataset.pathologies

[np.str_('Lung Opacity'),
 np.str_('Fracture'),
 np.str_('Nodule/Mass'),
 np.str_('Pneumothorax')]

In [6]:
# ============================================================
# Projection preparation
#
# Two possible high-dimensional feature spaces:
#
#   "Model embedding" -> DenseNet final hidden representation
#   "Dataset image"   -> flattened preprocessed X-ray
#
# The actual 2D projections are calculated lazily when selected.
# ============================================================

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, Isomap
import umap.umap_ as umap
import pacmap


projection_cache = {}


def get_feature_matrix(feature_space):
    """
    Return the high-dimensional representation corresponding
    to the selected feature space.
    """

    if feature_space == "Model embedding":
        return features

    elif feature_space == "Dataset image":
        return image_features

    else:
        raise ValueError(
            f"Unknown feature space: {feature_space}"
        )


def prepare_pca_features(feature_space):
    """
    PCA-reduce the selected high-dimensional feature space.

    PCA is used as a preprocessing step for the non-linear
    projection techniques.
    """

    cache_key = f"{feature_space}_pca"

    if cache_key in projection_cache:
        return projection_cache[cache_key]

    X = get_feature_matrix(feature_space)

    print(
        f"Preparing PCA representation for: "
        f"{feature_space}"
    )

    # Keep at most 50 dimensions before expensive
    # non-linear dimensionality reduction.
    n_components = min(
        50,
        X.shape[0],
        X.shape[1]
    )

    pca = PCA(
        n_components=n_components,
        random_state=RANDOM_STATE
    )

    X_pca = pca.fit_transform(X)

    projection_cache[cache_key] = {
        "data": X_pca,
        "pca": pca
    }

    print(
        f"{feature_space}: "
        f"{X.shape} -> {X_pca.shape}"
    )

    print(
        f"PCA variance retained: "
        f"{pca.explained_variance_ratio_.sum():.1%}"
    )

    return projection_cache[cache_key]


def calculate_projection(feature_space, method):
    """
    Calculate or retrieve a 2D projection for the selected
    feature space and dimensionality-reduction method.
    """

    cache_key = (
        f"{feature_space}_{method}"
    )

    if cache_key in projection_cache:
        return projection_cache[cache_key]

    print(
        f"\nComputing {method} "
        f"for {feature_space}..."
    )

    # --------------------------------------------------------
    # PCA
    # --------------------------------------------------------

    if method == "PCA":

        X = get_feature_matrix(feature_space)

        reducer = PCA(
            n_components=2,
            random_state=RANDOM_STATE
        )

        embedding = reducer.fit_transform(X)


    # --------------------------------------------------------
    # t-SNE
    # --------------------------------------------------------

    elif method == "t-SNE":

        X_pca = prepare_pca_features(
            feature_space
        )["data"]

        # t-SNE requires perplexity < number of samples.
        perplexity = min(
            30,
            max(5, (X_pca.shape[0] - 1) // 3)
        )

        reducer = TSNE(
            n_components=2,
            perplexity=perplexity,
            init="pca",
            learning_rate="auto",
            random_state=RANDOM_STATE
        )

        embedding = reducer.fit_transform(
            X_pca
        )


    # --------------------------------------------------------
    # UMAP
    # --------------------------------------------------------

    elif method == "UMAP":

        X_pca = prepare_pca_features(
            feature_space
        )["data"]

        n_neighbors = min(
            15,
            max(2, X_pca.shape[0] - 1)
        )

        reducer = umap.UMAP(
            n_components=2,
            n_neighbors=n_neighbors,
            min_dist=0.1,
            metric="euclidean",
            random_state=RANDOM_STATE
        )

        embedding = reducer.fit_transform(
            X_pca
        )


    # --------------------------------------------------------
    # Isomap
    # --------------------------------------------------------

    elif method == "Isomap":

        X_pca = prepare_pca_features(
            feature_space
        )["data"]

        n_neighbors = min(
            10,
            max(2, X_pca.shape[0] - 1)
        )

        reducer = Isomap(
            n_components=2,
            n_neighbors=n_neighbors,
            metric="euclidean",
        )

        embedding = reducer.fit_transform(
            X_pca
        )


    # --------------------------------------------------------
    # PaCMAP
    # --------------------------------------------------------

    elif method == "PaCMAP":

        X_pca = prepare_pca_features(
            feature_space
        )["data"]

        reducer = pacmap.PaCMAP(
            n_components=2,
            n_neighbors=10,
            MN_ratio=0.5,
            FP_ratio=2.0,
            random_state=RANDOM_STATE
        )

        embedding = reducer.fit_transform(
            X_pca
        )


    else:
        raise ValueError(
            f"Unknown projection method: {method}"
        )


    projection_cache[cache_key] = embedding

    print(
        f"{method} embedding: "
        f"{embedding.shape}"
    )

    return embedding

In [7]:
# Colors represent classifier outcomes, not ground-truth classes alone.
outcome_colors = {
    "TP": "#2E8B57",  # True positive
    "FN": "#E45756",  # False negative
    "TN": "#8295A5",  # True negative
    "FP": "#F2A541"   # False positive
}

outcome_names = {
    "TP": "True positive",
    "FN": "False negative",
    "TN": "True negative",
    "FP": "False positive"
}

In [8]:
dataset

{np.str_('Fracture'): {np.float32(0.0): 782, np.float32(1.0): 28},
 np.str_('Lung Opacity'): {np.float32(0.0): 589, np.float32(1.0): 221},
 np.str_('Nodule/Mass'): {np.float32(0.0): 689, np.float32(1.0): 121},
 np.str_('Pneumothorax'): {np.float32(0.0): 764, np.float32(1.0): 46}}


NIH_Google_Dataset num_samples=810 views=['PA'] data_aug=None

In [9]:
# Extract images (X) and labels (Y)
X_all = torch.stack([
    torch.from_numpy(dataset[i]["img"])
    for i in range(len(dataset))
])

Y_all = torch.tensor(dataset.labels, dtype=torch.float32)

In [10]:
# Use the model's pretrained classifier directly.
# No additional classifier is trained here: the DenseNet outputs
# extracted above are the predictions used for the analysis.

# Store predictions and metrics for every pathology
classifier_results = {}
metrics_rows = []

# Ground-truth labels come from the NIH dataset, while prediction
# columns come from the pretrained CheXpert DenseNet classifier.
dataset_label_indices = {
    name: dataset_pathologies.index(name)
    for name in pathologies
}

model_output_indices = {
    name: model.pathologies.index(name)
    for name in pathologies
}

for pathology in pathologies:

    print(f"\nEvaluating {pathology}")

    dataset_label_col = dataset_label_indices[pathology]
    model_output_col = model_output_indices[pathology]

    # Ground-truth labels for this pathology.
    y_all = labels[:, dataset_label_col]

    # Include only confirmed binary labels.
    # Exclude uncertain, missing, and other non-binary labels.
    valid_indices = np.flatnonzero(np.isin(y_all, [0, 1]))

    y_test = y_all[valid_indices].astype(int)

    # Use the pretrained model's classifier output directly.
    # TorchXRayVision's DenseNet returns pathology scores in [0, 1].
    y_prob = predictions[valid_indices, model_output_col]

    # Fixed threshold of 0.5 for this initial analysis.
    y_pred = (y_prob >= 0.5).astype(int)

    # Confusion-matrix categories for every evaluated sample.
    outcome = np.full(len(valid_indices), "", dtype=object)

    outcome[(y_test == 1) & (y_pred == 1)] = "TP"
    outcome[(y_test == 1) & (y_pred == 0)] = "FN"
    outcome[(y_test == 0) & (y_pred == 0)] = "TN"
    outcome[(y_test == 0) & (y_pred == 1)] = "FP"

    # Evaluate the pretrained model on all samples with confirmed
    # binary reference labels for this pathology.
    auc = roc_auc_score(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    bacc = balanced_accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )

    # Store a full-length prediction array so each prediction
    # can be mapped back to its position in the embedding.
    y_pred_full = np.full(len(labels), np.nan)
    y_prob_full = np.full(len(labels), np.nan)
    test_mask = np.zeros(len(labels), dtype=bool)
    outcome_full = np.full(len(labels), "", dtype=object)

    y_pred_full[valid_indices] = y_pred
    y_prob_full[valid_indices] = y_prob
    test_mask[valid_indices] = True
    outcome_full[valid_indices] = outcome

    # Save everything needed by the visualization cells.
    # "classifier" refers to the model's pretrained classifier layer;
    # no new classifier was fitted for this analysis.
    classifier_results[pathology] = {
        "classifier": model.classifier,
        "test_idx": valid_indices,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "test_mask": test_mask,
        "outcome": outcome_full,
        "y_pred_full": y_pred_full,
        "y_prob_full": y_prob_full,
        "auc": auc,
        "average_precision": ap,
        "balanced_accuracy": bacc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_test": len(valid_indices),
        "n_positive_test": int((y_test == 1).sum()),
        "n_negative_test": int((y_test == 0).sum())
    }

    # Store summary metrics.
    metrics_rows.append({
        "Pathology": pathology,
        "Test N": len(valid_indices),
        "Positive test": int((y_test == 1).sum()),
        "Negative test": int((y_test == 0).sum()),
        "ROC-AUC": auc,
        "Average precision": ap,
        "Balanced accuracy": bacc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

    print(
        f"Test samples: {len(valid_indices)} | "
        f"ROC-AUC: {auc:.3f} | "
        f"Average precision: {ap:.3f} | "
        f"Balanced accuracy: {bacc:.3f}"
    )

# Summary table
metrics_df = pd.DataFrame(metrics_rows)

print("\nPretrained model performance")
display(metrics_df.round(3))



Evaluating Lung Opacity
Test samples: 810 | ROC-AUC: 0.815 | Average precision: 0.666 | Balanced accuracy: 0.717

Evaluating Fracture
Test samples: 810 | ROC-AUC: 0.590 | Average precision: 0.073 | Balanced accuracy: 0.523

Evaluating Pneumothorax
Test samples: 810 | ROC-AUC: 0.711 | Average precision: 0.164 | Balanced accuracy: 0.597

Pretrained model performance


,Pathology,Test N,Positive test,Negative test,ROC-AUC,Average precision,Balanced accuracy,Precision,Recall,F1
0,Lung Opacity,810,221,589,0.815,0.666,0.717,0.429,0.864,0.574
1,Fracture,810,28,782,0.590,0.073,0.523,0.037,0.643,0.070
2,Pneumothorax,810,46,764,0.711,0.164,0.597,0.092,0.478,0.154


In [11]:
%matplotlib widget

import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.widgets import RectangleSelector

In [ ]:
# ============================================================
# Interactive DenseNet embedding explorer
# Select a region to inspect the corresponding chest X-rays.
# ============================================================


# Read sex metadata in dataset order. If the dataset does not expose it,
# points are marked as Unknown rather than inferring sex from the images.
def get_sex_values(dataset):
    metadata = getattr(dataset, "csv", None)
    if metadata is None or len(metadata) != len(dataset):
        return np.array(["Unknown"] * len(dataset), dtype=object)

    sex_column = next(
        (col for col in ["sex", "Patient Gender", "gender", "Sex"]
         if col in metadata.columns),
        None
    )
    if sex_column is None:
        return np.array(["Unknown"] * len(dataset), dtype=object)

    def normalize_sex(value):
        value = str(value).strip().lower()
        if value in {"m", "male", "1"}:
            return "Male"
        if value in {"f", "female", "0"}:
            return "Female"
        return "Unknown"

    return metadata[sex_column].map(normalize_sex).to_numpy()

sex_values = get_sex_values(dataset)
sex_markers = {"Male": "o", "Female": "^", "Unknown": "X"}

# Widgets for selecting the pathology and projection.
pathology_dropdown = widgets.Dropdown(
    options=pathologies,
    value=pathologies[0],
    description="Pathology:",
    layout=widgets.Layout(width="350px")
)

projection_dropdown = widgets.Dropdown(
    options=[
        "PCA",
        "t-SNE",
        "UMAP",
        "Isomap",
        "PaCMAP"
    ],
    value="UMAP",
    description="Projection:",
    layout=widgets.Layout(width="300px")
)

feature_space_dropdown = widgets.Dropdown(
    options=[
        "Model embedding",
        "Dataset image"
    ],
    value="Model embedding",
    description="Feature space:",
    layout=widgets.Layout(width="400px")
)

prediction_dropdown = widgets.Dropdown(
    options=[
        "All",
        "Predicted positive",
        "Predicted negative",
        "False positive",
        "False negative",
        "Both wrong prediction types"
    ],
    value="All",
    description="Prediction:",
    layout=widgets.Layout(width="350px")
)

max_images_slider = widgets.IntSlider(
    value=12,
    min=4,
    max=24,
    step=4,
    description="Show images:",
    continuous_update=False,
    layout=widgets.Layout(width="350px")
)

plot_button = widgets.Button(
    description="Create selection plot",
    button_style="primary",
    icon="refresh"
)

plot_output = widgets.Output()
image_output = widgets.Output()

# Keep references to the interactive objects alive.
interactive_state = {
    "figure": None,
    "axes": None,
    "selector": None,
    "selected_indices": []
}


def show_selected_xrays(indices, pathology):
    """
    Display the actual X-rays corresponding to selected
    embedding points.
    """

    with image_output:
        clear_output(wait=True)

        if len(indices) == 0:
            print(
                "No images selected. Draw a rectangle around "
                "points in the embedding."
            )
            return

        # Limit the number of images displayed at once.
        n_show = min(
            len(indices),
            max_images_slider.value
        )

        selected_to_show = indices[:n_show]

        print(
            f"Selected {len(indices)} images. "
            f"Displaying {n_show}."
        )

        # Look up the selected pathology label.
        label_idx = dataset_pathologies.index(pathology)

        ncols = 4
        nrows = int(np.ceil(n_show / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(12, 3.5 * nrows),
            squeeze=False
        )

        for ax in axes.flat:
            ax.axis("off")

        for ax, idx in zip(
            axes.flat,
            selected_to_show
        ):

            # The dataset index matches the embedding row.
            sample = dataset[int(idx)]

            img = np.asarray(sample["img"]).squeeze()

            # Display robustly across different X-ray value ranges.
            vmin, vmax = np.percentile(
                img[np.isfinite(img)],
                [1, 99]
            )

            ax.imshow(
                img,
                cmap="gray",
                vmin=vmin,
                vmax=vmax
            )

            # Ground-truth pathology label.
            label_value = labels[
                int(idx),
                label_idx
            ]

            if np.isfinite(label_value):
                if label_value == 1:
                    gt_text = "Positive"
                elif label_value == 0:
                    gt_text = "Negative"
                else:
                    gt_text = "Unknown"
            else:
                gt_text = "Unknown"

            # If a held-out classifier result exists, show it.
            result = classifier_results.get(pathology)

            if (
                result is not None
                and result["test_mask"][int(idx)]
            ):
                outcome = result["outcome"][int(idx)]

                if outcome:
                    outcome_text = outcome
                else:
                    outcome_text = "Not classified"
            else:
                outcome_text = "Not in test set"

            ax.set_title(
                f"Dataset index: {idx}\n"
                f"Ground truth: {gt_text}\n"
                f"Classifier: {outcome_text}",
                fontsize=9
            )

        fig.suptitle(
            f"Selected X-rays — {pathology}",
            fontsize=14
        )

        plt.tight_layout()
        plt.show()


def create_selection_plot(_=None):

    with plot_output:
        clear_output(wait=True)

        pathology = pathology_dropdown.value
        method = projection_dropdown.value
        feature_space = feature_space_dropdown.value

        # Calculate the requested projection if it has not already
        # been calculated.
        embedding = calculate_projection(
            feature_space,
            method
        )

        label_idx = dataset_pathologies.index(pathology)
        y = labels[:, label_idx]

        # Filter by the selected prediction type, using the existing
        # held-out classifier results. Samples outside the test set
        # are hidden when a prediction filter is selected.
        prediction_filter = prediction_dropdown.value
        prediction_mask = np.ones(len(dataset), dtype=bool)
        result = classifier_results.get(pathology)

        if prediction_filter != "All":
            prediction_mask[:] = False
            if result is not None:
                if prediction_filter == "Predicted positive":
                    prediction_mask = result["test_mask"] & (result["y_pred_full"] == 1)
                elif prediction_filter == "Predicted negative":
                    prediction_mask = result["test_mask"] & (result["y_pred_full"] == 0)
                elif prediction_filter == "False positive":
                    prediction_mask = result["outcome"] == "FP"
                elif prediction_filter == "False negative":
                    prediction_mask = result["outcome"] == "FN"
                elif prediction_filter == "Both wrong prediction types":
                    prediction_mask = np.isin(result["outcome"], ["FP", "FN"])

        # Identify samples with known binary labels.
        positive = np.isfinite(y) & (y == 1) & prediction_mask
        negative = np.isfinite(y) & (y == 0) & prediction_mask
        unknown = ~(np.isfinite(y) & ((y == 1) | (y == 0))) & prediction_mask

        # Create a new plot.
        fig, ax = plt.subplots(figsize=(11, 8))

        # Color encodes pathology label; marker shape encodes sex.
        label_groups = [
            (negative, "#8295A5", "Negative"),
            (positive, "#E45756", "Positive"),
            (unknown, "#D4A72C", "Unknown label")
        ]

        for label_mask, color, label_name in label_groups:
            for sex_name, marker in sex_markers.items():
                mask = label_mask & (sex_values == sex_name)
                if mask.any():
                    ax.scatter(
                        embedding[mask, 0],
                        embedding[mask, 1],
                        c=color,
                        marker=marker,
                        s=30,
                        alpha=0.75,
                        edgecolors="white" if label_name == "Positive" else "none",
                        linewidths=0.3,
                        label=f"{label_name} / {sex_name}"
                    )

        ax.set_title(
            f"{pathology} — {feature_space} — {method}\n"
            f"{prediction_filter}\n"
            "Draw a rectangle around a region to inspect its X-rays"
        )

        ax.set_xlabel(
            f"{method} dimension 1"
        )

        ax.set_ylabel(
            f"{method} dimension 2"
        )

        ax.grid(alpha=0.2)

        color_legend = ax.legend(
            handles=[
                Line2D([0], [0], marker="o", color="w",
                       markerfacecolor="#8295A5", markersize=7, label="Negative"),
                Line2D([0], [0], marker="o", color="w",
                       markerfacecolor="#E45756", markersize=7, label="Positive"),
                Line2D([0], [0], marker="o", color="w",
                       markerfacecolor="#D4A72C", markersize=7, label="Unknown label")
            ],
            title="Pathology label",
            loc="upper left"
        )
        ax.add_artist(color_legend)
        ax.legend(
            handles=[
                Line2D([0], [0], marker=marker, color="gray", linestyle="None",
                       markersize=7, label=sex_name)
                for sex_name, marker in sex_markers.items()
            ],
            title="Patient sex (shape)",
            loc="upper right"
        )

        # Keep the selection tool available after displaying the plot.
        selector = RectangleSelector(
            ax,
            onselect=lambda eclick, erelease: on_region_selected(
                eclick,
                erelease,
                embedding,
                pathology
            ),
            useblit=True,
            button=[1],
            minspanx=0.01,
            minspany=0.01,
            spancoords="data",
            interactive=True
        )

        interactive_state["figure"] = fig
        interactive_state["axes"] = ax
        interactive_state["selector"] = selector
        interactive_state["selected_indices"] = []

        plt.show()


def on_region_selected(
    eclick,
    erelease,
    embedding,
    pathology
):
    """
    Find embedding points inside the selected rectangle
    and display their corresponding X-rays.
    """

    if (
        eclick.xdata is None
        or eclick.ydata is None
        or erelease.xdata is None
        or erelease.ydata is None
    ):
        return

    # Rectangle boundaries.
    xmin, xmax = sorted([
        eclick.xdata,
        erelease.xdata
    ])

    ymin, ymax = sorted([
        eclick.ydata,
        erelease.ydata
    ])

    # Identify all samples whose 2D coordinates lie
    # inside the rectangle.
    selected_mask = (
        (embedding[:, 0] >= xmin)
        & (embedding[:, 0] <= xmax)
        & (embedding[:, 1] >= ymin)
        & (embedding[:, 1] <= ymax)
    )

    indices = np.flatnonzero(selected_mask)

    interactive_state["selected_indices"] = indices

    show_selected_xrays(
        indices.tolist(),
        pathology
    )


# Connect the button.
plot_button.on_click(create_selection_plot)

# Display the controls and output areas.
display(
    widgets.VBox([

        widgets.HBox([
            feature_space_dropdown,
            projection_dropdown
        ]),

        widgets.HBox([
            pathology_dropdown,
            prediction_dropdown
        ]),

        max_images_slider,

        plot_button,

        plot_output,

        image_output
    ])
)

# Create the first plot automatically.
create_selection_plot()

In [ ]:
# ============================================================
# Second interactive plot: model score by patient age
# X-axis = age; Y-axis = model output score; color = predicted pathology; shape = sex
# ============================================================

# Find the age column in the dataset metadata.
metadata = getattr(dataset, "csv", None)
age_column = next(
    (col for col in ["Patient Age", "age", "Age", "patient_age"]
     if metadata is not None and col in metadata.columns),
    None
)

if metadata is None or len(metadata) != len(dataset) or age_column is None:
    print("Age metadata is not available in dataset.csv; age plot cannot be created.")
else:
    age_values = pd.to_numeric(metadata[age_column], errors="coerce").to_numpy()

    age_pathology_dropdown = widgets.Dropdown(
        options=pathologies,
        value=pathologies[0],
        description="Pathology:",
        layout=widgets.Layout(width="350px")
    )
    age_plot_button = widgets.Button(
        description="Create age plot",
        button_style="primary",
        icon="refresh"
    )
    age_plot_output = widgets.Output()

    def create_age_plot(_=None):
        with age_plot_output:
            clear_output(wait=True)

            pathology = age_pathology_dropdown.value
            label_idx = dataset_pathologies.index(pathology)

            # Sort samples by age; omit samples with missing/non-numeric ages.
            valid = np.isfinite(age_values)
            sorted_indices = np.flatnonzero(valid)
            sorted_indices = sorted_indices[
                np.argsort(age_values[sorted_indices], kind="stable")
            ]

            # Use the model's output for the selected pathology.
            # Threshold at 0.5 to display predicted Positive/Negative.
            predicted_positive = predictions[:, label_idx] >= 0.5
            prediction_groups = [
                (predicted_positive, "#E45756", "Predicted positive"),
                (~predicted_positive, "#8295A5", "Predicted negative")
            ]

            fig, ax = plt.subplots(figsize=(12, 6))

            # Model output score for the selected pathology.
            confidence = predictions[:, label_idx]

            for pred_mask, color, pred_name in prediction_groups:
                for sex_name, marker in sex_markers.items():
                    mask = (
                        np.isin(np.arange(len(dataset)), sorted_indices)
                        & pred_mask
                        & (sex_values == sex_name)
                    )
                    idx = np.flatnonzero(mask)
                    if len(idx):
                        ax.scatter(
                            age_values[idx],
                            confidence[idx],
                            c=color,
                            marker=marker,
                            s=28,
                            alpha=0.75,
                            edgecolors="none",
                            label=f"{pred_name} / {sex_name}"
                        )

            ax.set_title(f"Model score by patient age — {pathology}")
            ax.set_xlabel("Patient age")
            ax.set_ylabel("Model output score (confidence score)")
            ax.grid(alpha=0.2)

            color_legend = ax.legend(
                handles=[
                    Line2D([0], [0], marker="o", color="w",
                           markerfacecolor="#E45756", markersize=7,
                           label="Predicted positive"),
                    Line2D([0], [0], marker="o", color="w",
                           markerfacecolor="#8295A5", markersize=7,
                           label="Predicted negative")
                ],
                title="Model prediction (color)",
                loc="upper left"
            )
            ax.add_artist(color_legend)
            ax.legend(
                handles=[
                    Line2D([0], [0], marker=marker, color="gray",
                           linestyle="None", markersize=7, label=sex_name)
                    for sex_name, marker in sex_markers.items()
                ],
                title="Patient sex (shape)",
                loc="upper right"
            )

            plt.tight_layout()
            plt.show()

    age_plot_button.on_click(create_age_plot)
    display(widgets.VBox([
        widgets.HBox([age_pathology_dropdown, age_plot_button]),
        age_plot_output
    ]))
    create_age_plot()
